# FIT5217 A2 Task 1 Colab Training Notebook

This notebook trains the Task 1.2 baseline RNN encoder-decoder and the Task 1.3 Bahdanau attention model.

Recommended runtime: **GPU L4**. T1 is expected to use roughly **30-50 Colab units** depending on queue, GPU allocation, and how many epochs you run.

Expected outputs are saved back to Google Drive under:

`/content/drive/MyDrive/fit5217_a2/checkpoints/`

The preprocessed data is expected to already exist in Google Drive under:

`/content/drive/MyDrive/fit5217_a2/outputs/`


In [ ]:
# GPU check
!nvidia-smi

import torch

if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_id)
    total_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name}")
    print(f"Total VRAM: {total_gb:.1f} GB")
else:
    print("No CUDA GPU detected. Please switch Runtime > Change runtime type > GPU.")

print("Recommendation: use an L4 runtime for Task 1; budget target is around 30-50 Colab units.")


In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Clone repository
from pathlib import Path
import os

REPO_URL = 'https://github.com/jche0572/fit5217-a2.git'
REPO_DIR = Path('/content/fit5217-a2')

if REPO_DIR.exists():
    print(f'Repository already exists: {REPO_DIR}')
else:
    %cd /content
    !git clone {REPO_URL}

%cd /content/fit5217-a2
!git rev-parse --short HEAD


In [ ]:
# Install dependencies
%cd /content/fit5217-a2
!pip install -q -r requirements-colab.txt

import torch
import transformers
import sacrebleu
import nltk
import bert_score

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sacrebleu:', sacrebleu.__version__)
print('nltk:', nltk.__version__)
print('bert_score import: OK')


In [ ]:
# Prepare Drive-backed outputs and verify preprocessed data
from pathlib import Path
import pickle
import shutil
import os
import sys

REPO_DIR = Path('/content/fit5217-a2')
DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
LOCAL_OUTPUTS = REPO_DIR / 'outputs'

DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# Use a symlink so scripts and modules can keep using ./outputs while all artifacts live on Drive.
if LOCAL_OUTPUTS.exists() and not LOCAL_OUTPUTS.is_symlink():
    backup = REPO_DIR / 'outputs_repo_backup'
    if not backup.exists():
        shutil.move(str(LOCAL_OUTPUTS), str(backup))
        print(f'Moved repository outputs directory to {backup}')
    else:
        shutil.rmtree(LOCAL_OUTPUTS)

if LOCAL_OUTPUTS.is_symlink() or LOCAL_OUTPUTS.exists():
    if LOCAL_OUTPUTS.resolve() != DRIVE_OUTPUTS.resolve():
        LOCAL_OUTPUTS.unlink()

if not LOCAL_OUTPUTS.exists():
    LOCAL_OUTPUTS.symlink_to(DRIVE_OUTPUTS, target_is_directory=True)

print('Local outputs:', LOCAL_OUTPUTS, '->', LOCAL_OUTPUTS.resolve())
print('Drive checkpoints:', DRIVE_CHECKPOINTS)

required_files = [
    DRIVE_OUTPUTS / 'vocab.pkl',
    DRIVE_OUTPUTS / 'processed' / 'train_ids.pkl',
    DRIVE_OUTPUTS / 'processed' / 'dev_ids.pkl',
    DRIVE_OUTPUTS / 'processed' / 'test_ids.pkl',
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required Drive files:\n' + '\n'.join(missing))

with open(DRIVE_OUTPUTS / 'vocab.pkl', 'rb') as f:
    vocab_state = pickle.load(f)

print('Vocab size:', len(vocab_state['word2idx']))
for split in ['train', 'dev', 'test']:
    path = DRIVE_OUTPUTS / 'processed' / f'{split}_ids.pkl'
    with open(path, 'rb') as f:
        rows = pickle.load(f)
    print(f'{split}: {len(rows):,} examples')


## Train T1.2 Baseline GRU Encoder-Decoder

This section trains the baseline model without attention. Checkpoints are written to Drive after every epoch, and the cell resumes from `last.pt` if it exists.


In [ ]:
# Train T1.2 baseline with Drive checkpoint resume
import sys
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.dataset import RecipeDataset, collate_fn, load_vocab_size
from src.models.rnn_baseline import Seq2Seq
from src.training import EarlyStopping, eval_epoch, load_checkpoint, make_criterion, save_checkpoint, train_epoch
from src.utils import get_device, set_seed

set_seed(42)
DEVICE = get_device()
print('Device:', DEVICE)

# Hyperparameter rationale:
# 10 epochs is a budget-aware ceiling for 162k train rows on L4. Early stopping with patience=3
# usually stops before this if dev loss plateaus. Batch 128 keeps VRAM modest and throughput stable.
BASELINE_CONFIG = {
    'epochs': 10,
    'batch_size': 128,
    'max_tgt_len': 80,
    'embed_size': 128,
    'hidden_size': 256,
    'num_layers': 1,
    'dropout': 0.1,
    'lr': 1e-3,
    'clip': 1.0,
    'patience': 3,
}

DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
BASELINE_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 't1_baseline'
BASELINE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'train_ids.pkl', max_tgt_len=BASELINE_CONFIG['max_tgt_len'])
dev_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'dev_ids.pkl', max_tgt_len=BASELINE_CONFIG['max_tgt_len'])
train_loader = DataLoader(train_dataset, batch_size=BASELINE_CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
dev_loader = DataLoader(dev_dataset, batch_size=BASELINE_CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

vocab_size = load_vocab_size(DRIVE_OUTPUTS / 'vocab.pkl')
baseline_model = Seq2Seq(
    vocab_size=vocab_size,
    embed_size=BASELINE_CONFIG['embed_size'],
    hidden_size=BASELINE_CONFIG['hidden_size'],
    num_layers=BASELINE_CONFIG['num_layers'],
    dropout=BASELINE_CONFIG['dropout'],
).to(DEVICE)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=BASELINE_CONFIG['lr'])
criterion = make_criterion()
stopper = EarlyStopping(patience=BASELINE_CONFIG['patience'])

last_path = BASELINE_CKPT_DIR / 'last.pt'
best_path = BASELINE_CKPT_DIR / 'best.pt'
train_losses = []
dev_losses = []
start_epoch = 1

if last_path.exists():
    state = load_checkpoint(last_path, baseline_model, baseline_optimizer, map_location=DEVICE)
    train_losses = list(state.get('train_losses', []))
    dev_losses = list(state.get('dev_losses', []))
    start_epoch = int(state.get('epoch', 0)) + 1
    if dev_losses:
        stopper.best_loss = min(dev_losses)
    print(f'Resumed baseline from {last_path} at epoch {start_epoch}')

full_config = dict(BASELINE_CONFIG)
full_config.update({'vocab_size': vocab_size, 'seed': 42, 'cell': 'GRU', 'bidirectional': False, 'attention': None})
training_start = time.time()

for epoch in range(start_epoch, BASELINE_CONFIG['epochs'] + 1):
    epoch_start = time.time()
    train_loss = train_epoch(baseline_model, train_loader, baseline_optimizer, criterion, DEVICE, clip=BASELINE_CONFIG['clip'])
    dev_loss = eval_epoch(baseline_model, dev_loader, criterion, DEVICE)
    elapsed = time.time() - epoch_start
    train_losses.append(train_loss)
    dev_losses.append(dev_loss)
    print(f'[baseline] epoch={epoch} train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} elapsed={elapsed/60:.1f} min')

    save_checkpoint(last_path, baseline_model, baseline_optimizer, epoch, train_losses, dev_losses, full_config)
    if dev_loss <= min(dev_losses):
        save_checkpoint(best_path, baseline_model, baseline_optimizer, epoch, train_losses, dev_losses, full_config)
        print(f'[baseline] saved best checkpoint: {best_path}')

    if stopper.step(dev_loss):
        print(f'[baseline] early stopping at epoch {epoch}; best_dev_loss={stopper.best_loss:.4f}')
        break

BASELINE_TRAIN_SECONDS = time.time() - training_start
print(f'Baseline total training time in this run: {BASELINE_TRAIN_SECONDS/60:.1f} min')
print('Baseline checkpoints:', BASELINE_CKPT_DIR)


In [ ]:
# T1.2 baseline inference on the full test split + metrics
import json
import pickle
import re
import sys
import time
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import nltk
import numpy as np
import torch
from bert_score import score as bert_score
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from torch.utils.data import DataLoader

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.dataset import RecipeDataset, collate_fn, load_vocab_size
from src.models.rnn_baseline import Seq2Seq
from src.preprocessing import EOS_ID, PAD_ID, SOS_ID
from src.training import load_checkpoint
from src.utils import get_device

DEVICE = get_device()
DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
BASELINE_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 't1_baseline'
PRED_DIR = DRIVE_OUTPUTS / 'predictions'
PRED_DIR.mkdir(parents=True, exist_ok=True)

with open(DRIVE_OUTPUTS / 'vocab.pkl', 'rb') as f:
    vocab_state = pickle.load(f)
idx2word = {idx: tok for tok, idx in vocab_state['word2idx'].items()}

def decode_ids(ids):
    tokens = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id in (PAD_ID, SOS_ID):
            continue
        if token_id == EOS_ID:
            break
        tokens.append(idx2word.get(token_id, '<UNK>'))
    return ' '.join(tokens)

def patch_eval_metrics():
    eval_path = REPO_DIR / 'eval_metrics.py'
    text = eval_path.read_text()
    patched = re.sub(r'nltk_cache_dir\s*=\s*\[YOUR_DIR\]', "nltk_cache_dir = '/content/nltk_data'", text)
    if patched != text:
        eval_path.write_text(patched)
    Path('/content/nltk_data').mkdir(parents=True, exist_ok=True)

def compute_metrics(gold_recipes, pred_recipes):
    nltk.download('punkt', download_dir='/content/nltk_data', quiet=True)
    nltk.download('wordnet', download_dir='/content/nltk_data', quiet=True)
    nltk.download('omw-1.4', download_dir='/content/nltk_data', quiet=True)
    nltk.download('punkt_tab', download_dir='/content/nltk_data', quiet=True)
    if '/content/nltk_data' not in nltk.data.path:
        nltk.data.path.append('/content/nltk_data')
    refs = [[nltk.word_tokenize(gold)] for gold in gold_recipes]
    hyps = [nltk.word_tokenize(pred) for pred in pred_recipes]
    sm = SmoothingFunction().method4
    bleu4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=sm)
    meteor = float(np.mean([meteor_score(ref, hyp) for ref, hyp in zip(refs, hyps)]))
    _, _, f1 = bert_score(pred_recipes, gold_recipes, lang='en', verbose=False)
    return {'BLEU-4': float(bleu4), 'METEOR': meteor, 'BERTScore': float(f1.cpu().numpy().mean())}

# Load model using the saved checkpoint config, so inference still works if training hyperparameters are edited.
baseline_state = torch.load(BASELINE_CKPT_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
baseline_config = baseline_state.get('config', {})
vocab_size = baseline_config.get('vocab_size', load_vocab_size(DRIVE_OUTPUTS / 'vocab.pkl'))
baseline_model = Seq2Seq(
    vocab_size=vocab_size,
    embed_size=baseline_config.get('embed_size', 128),
    hidden_size=baseline_config.get('hidden_size', 256),
    num_layers=baseline_config.get('num_layers', 1),
    dropout=baseline_config.get('dropout', 0.1),
).to(DEVICE)
load_checkpoint(BASELINE_CKPT_DIR / 'best.pt', baseline_model, map_location=DEVICE)
baseline_model.eval()

test_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'test_ids.pkl', max_tgt_len=80)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

predictions = []
gold_recipes = []
start = time.time()
with torch.no_grad():
    for batch in test_loader:
        src = batch['src'].to(DEVICE)
        src_lengths = batch['src_lengths'].to(DEVICE)
        generated = baseline_model.generate(src, src_lengths, max_len=80)
        predictions.extend(decode_ids(row) for row in generated.cpu())
        gold_recipes.extend(decode_ids(row) for row in batch['tgt'])

BASELINE_TEST_SECONDS = time.time() - start
out_path = PRED_DIR / 't1_baseline_test.json'
with open(out_path, 'w') as f:
    json.dump([{'gold': g, 'prediction': p} for g, p in zip(gold_recipes, predictions)], f, indent=2)
print(f'Saved baseline predictions: {out_path}')
print(f'Baseline inference time: {BASELINE_TEST_SECONDS/60:.1f} min')

patch_eval_metrics()
from eval_metrics import evaluate2
print('Teacher eval_metrics.evaluate2 output:')
evaluate2(gold_recipes, predictions)
BASELINE_METRICS = compute_metrics(gold_recipes, predictions)
print('Baseline metrics dict:', BASELINE_METRICS)


## Train T1.3 Bahdanau Attention Model

This section trains the manually implemented Bahdanau attention model. Checkpoints are written to Drive after every epoch, and the cell resumes from `last.pt` if it exists.


In [ ]:
# Train T1.3 attention with Drive checkpoint resume
import sys
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.dataset import RecipeDataset, collate_fn, load_vocab_size
from src.models.rnn_attention import Seq2SeqAttention
from src.training import EarlyStopping, eval_epoch, load_checkpoint, make_criterion, save_checkpoint, train_epoch
from src.utils import get_device, set_seed

set_seed(42)
DEVICE = get_device()
print('Device:', DEVICE)

# Hyperparameter rationale:
# Attention is slower because it loops over target steps and computes a source distribution each step.
# Keep the same model size as baseline for a fair comparison; use 8 epochs as a budget-aware ceiling.
ATTENTION_CONFIG = {
    'epochs': 8,
    'batch_size': 128,
    'max_tgt_len': 80,
    'embed_size': 128,
    'hidden_size': 256,
    'num_layers': 1,
    'dropout': 0.1,
    'lr': 1e-3,
    'clip': 1.0,
    'patience': 3,
}

DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
ATTENTION_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 't1_attention'
ATTENTION_CKPT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'train_ids.pkl', max_tgt_len=ATTENTION_CONFIG['max_tgt_len'])
dev_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'dev_ids.pkl', max_tgt_len=ATTENTION_CONFIG['max_tgt_len'])
train_loader = DataLoader(train_dataset, batch_size=ATTENTION_CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
dev_loader = DataLoader(dev_dataset, batch_size=ATTENTION_CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

vocab_size = load_vocab_size(DRIVE_OUTPUTS / 'vocab.pkl')
attention_model = Seq2SeqAttention(
    vocab_size=vocab_size,
    embed_size=ATTENTION_CONFIG['embed_size'],
    hidden_size=ATTENTION_CONFIG['hidden_size'],
    num_layers=ATTENTION_CONFIG['num_layers'],
    dropout=ATTENTION_CONFIG['dropout'],
).to(DEVICE)
attention_optimizer = torch.optim.Adam(attention_model.parameters(), lr=ATTENTION_CONFIG['lr'])
criterion = make_criterion()
stopper = EarlyStopping(patience=ATTENTION_CONFIG['patience'])

last_path = ATTENTION_CKPT_DIR / 'last.pt'
best_path = ATTENTION_CKPT_DIR / 'best.pt'
train_losses = []
dev_losses = []
start_epoch = 1

if last_path.exists():
    state = load_checkpoint(last_path, attention_model, attention_optimizer, map_location=DEVICE)
    train_losses = list(state.get('train_losses', []))
    dev_losses = list(state.get('dev_losses', []))
    start_epoch = int(state.get('epoch', 0)) + 1
    if dev_losses:
        stopper.best_loss = min(dev_losses)
    print(f'Resumed attention model from {last_path} at epoch {start_epoch}')

full_config = dict(ATTENTION_CONFIG)
full_config.update({'vocab_size': vocab_size, 'seed': 42, 'cell': 'GRU', 'bidirectional': False, 'attention': 'bahdanau_manual'})
training_start = time.time()

for epoch in range(start_epoch, ATTENTION_CONFIG['epochs'] + 1):
    epoch_start = time.time()
    train_loss = train_epoch(attention_model, train_loader, attention_optimizer, criterion, DEVICE, clip=ATTENTION_CONFIG['clip'])
    dev_loss = eval_epoch(attention_model, dev_loader, criterion, DEVICE)
    elapsed = time.time() - epoch_start
    train_losses.append(train_loss)
    dev_losses.append(dev_loss)
    print(f'[attention] epoch={epoch} train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} elapsed={elapsed/60:.1f} min')

    save_checkpoint(last_path, attention_model, attention_optimizer, epoch, train_losses, dev_losses, full_config)
    if dev_loss <= min(dev_losses):
        save_checkpoint(best_path, attention_model, attention_optimizer, epoch, train_losses, dev_losses, full_config)
        print(f'[attention] saved best checkpoint: {best_path}')

    if stopper.step(dev_loss):
        print(f'[attention] early stopping at epoch {epoch}; best_dev_loss={stopper.best_loss:.4f}')
        break

ATTENTION_TRAIN_SECONDS = time.time() - training_start
print(f'Attention total training time in this run: {ATTENTION_TRAIN_SECONDS/60:.1f} min')
print('Attention checkpoints:', ATTENTION_CKPT_DIR)


In [ ]:
# T1.3 attention inference on the full test split + metrics
import json
import pickle
import re
import sys
import time
from pathlib import Path

import nltk
import numpy as np
import torch
from bert_score import score as bert_score
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from torch.utils.data import DataLoader

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.dataset import RecipeDataset, collate_fn, load_vocab_size
from src.models.rnn_attention import Seq2SeqAttention
from src.preprocessing import EOS_ID, PAD_ID, SOS_ID
from src.training import load_checkpoint
from src.utils import get_device

DEVICE = get_device()
DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
ATTENTION_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 't1_attention'
PRED_DIR = DRIVE_OUTPUTS / 'predictions'
PRED_DIR.mkdir(parents=True, exist_ok=True)

with open(DRIVE_OUTPUTS / 'vocab.pkl', 'rb') as f:
    vocab_state = pickle.load(f)
idx2word = {idx: tok for tok, idx in vocab_state['word2idx'].items()}

def decode_ids(ids):
    tokens = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id in (PAD_ID, SOS_ID):
            continue
        if token_id == EOS_ID:
            break
        tokens.append(idx2word.get(token_id, '<UNK>'))
    return ' '.join(tokens)

def patch_eval_metrics():
    eval_path = REPO_DIR / 'eval_metrics.py'
    text = eval_path.read_text()
    patched = re.sub(r'nltk_cache_dir\s*=\s*\[YOUR_DIR\]', "nltk_cache_dir = '/content/nltk_data'", text)
    if patched != text:
        eval_path.write_text(patched)
    Path('/content/nltk_data').mkdir(parents=True, exist_ok=True)

def compute_metrics(gold_recipes, pred_recipes):
    nltk.download('punkt', download_dir='/content/nltk_data', quiet=True)
    nltk.download('wordnet', download_dir='/content/nltk_data', quiet=True)
    nltk.download('omw-1.4', download_dir='/content/nltk_data', quiet=True)
    nltk.download('punkt_tab', download_dir='/content/nltk_data', quiet=True)
    if '/content/nltk_data' not in nltk.data.path:
        nltk.data.path.append('/content/nltk_data')
    refs = [[nltk.word_tokenize(gold)] for gold in gold_recipes]
    hyps = [nltk.word_tokenize(pred) for pred in pred_recipes]
    sm = SmoothingFunction().method4
    bleu4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=sm)
    meteor = float(np.mean([meteor_score(ref, hyp) for ref, hyp in zip(refs, hyps)]))
    _, _, f1 = bert_score(pred_recipes, gold_recipes, lang='en', verbose=False)
    return {'BLEU-4': float(bleu4), 'METEOR': meteor, 'BERTScore': float(f1.cpu().numpy().mean())}

attention_state = torch.load(ATTENTION_CKPT_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
attention_config = attention_state.get('config', {})
vocab_size = attention_config.get('vocab_size', load_vocab_size(DRIVE_OUTPUTS / 'vocab.pkl'))
attention_model = Seq2SeqAttention(
    vocab_size=vocab_size,
    embed_size=attention_config.get('embed_size', 128),
    hidden_size=attention_config.get('hidden_size', 256),
    num_layers=attention_config.get('num_layers', 1),
    dropout=attention_config.get('dropout', 0.1),
).to(DEVICE)
load_checkpoint(ATTENTION_CKPT_DIR / 'best.pt', attention_model, map_location=DEVICE)
attention_model.eval()

test_dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'test_ids.pkl', max_tgt_len=80)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))

predictions = []
gold_recipes = []
start = time.time()
with torch.no_grad():
    for batch in test_loader:
        src = batch['src'].to(DEVICE)
        src_lengths = batch['src_lengths'].to(DEVICE)
        generated = attention_model.generate(src, src_lengths, max_len=80)
        predictions.extend(decode_ids(row) for row in generated.cpu())
        gold_recipes.extend(decode_ids(row) for row in batch['tgt'])

ATTENTION_TEST_SECONDS = time.time() - start
out_path = PRED_DIR / 't1_attention_test.json'
with open(out_path, 'w') as f:
    json.dump([{'gold': g, 'prediction': p} for g, p in zip(gold_recipes, predictions)], f, indent=2)
print(f'Saved attention predictions: {out_path}')
print(f'Attention inference time: {ATTENTION_TEST_SECONDS/60:.1f} min')

patch_eval_metrics()
from eval_metrics import evaluate2
print('Teacher eval_metrics.evaluate2 output:')
evaluate2(gold_recipes, predictions)
ATTENTION_METRICS = compute_metrics(gold_recipes, predictions)
print('Attention metrics dict:', ATTENTION_METRICS)


In [ ]:
# Generate and display two attention heatmaps
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader

REPO_DIR = Path('/content/fit5217-a2')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.dataset import RecipeDataset, collate_fn, load_vocab_size
from src.models.rnn_attention import Seq2SeqAttention
from src.preprocessing import EOS_ID, PAD_ID, SOS_ID
from src.training import load_checkpoint
from src.utils import get_device

DEVICE = get_device()
DRIVE_ROOT = Path('/content/drive/MyDrive/fit5217_a2')
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
ATTENTION_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 't1_attention'
HEATMAP_DIR = DRIVE_OUTPUTS / 'attention_heatmaps'
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

with open(DRIVE_OUTPUTS / 'vocab.pkl', 'rb') as f:
    vocab_state = pickle.load(f)
idx2word = {idx: tok for tok, idx in vocab_state['word2idx'].items()}

def ids_to_tokens(ids, stop_at_eos=False):
    tokens = []
    for token_id in ids.detach().cpu().tolist():
        token_id = int(token_id)
        if token_id in (PAD_ID, SOS_ID):
            continue
        if token_id == EOS_ID:
            if stop_at_eos:
                break
            continue
        tokens.append(idx2word.get(token_id, '<UNK>'))
    return tokens

def save_heatmap(attention, src_tokens, generated_tokens, path):
    rows = max(len(generated_tokens), 1)
    cols = max(len(src_tokens), 1)
    matrix = attention[:rows, :cols].detach().cpu().numpy()
    fig_width = max(8, min(18, cols * 0.35))
    fig_height = max(4, min(16, rows * 0.28))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    im = ax.imshow(matrix, aspect='auto', cmap='viridis')
    ax.set_xticks(range(cols))
    ax.set_xticklabels(src_tokens or ['<empty>'], rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(rows))
    ax.set_yticklabels(generated_tokens or ['<empty>'], fontsize=8)
    ax.set_xlabel('Ingredient tokens')
    ax.set_ylabel('Generated recipe tokens')
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

attention_state = torch.load(ATTENTION_CKPT_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
attention_config = attention_state.get('config', {})
vocab_size = attention_config.get('vocab_size', load_vocab_size(DRIVE_OUTPUTS / 'vocab.pkl'))
attention_model = Seq2SeqAttention(
    vocab_size=vocab_size,
    embed_size=attention_config.get('embed_size', 128),
    hidden_size=attention_config.get('hidden_size', 256),
    num_layers=attention_config.get('num_layers', 1),
    dropout=attention_config.get('dropout', 0.1),
).to(DEVICE)
load_checkpoint(ATTENTION_CKPT_DIR / 'best.pt', attention_model, map_location=DEVICE)
attention_model.eval()

dataset = RecipeDataset(DRIVE_OUTPUTS / 'processed' / 'test_ids.pkl', max_tgt_len=80, max_items=2)
loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))
src = batch['src'].to(DEVICE)
src_lengths = batch['src_lengths'].to(DEVICE)

with torch.no_grad():
    generated, attention = attention_model.generate(src, src_lengths, max_len=80, return_attention=True)

for i in range(src.size(0)):
    src_tokens = ids_to_tokens(batch['src'][i])
    generated_tokens = ids_to_tokens(generated[i], stop_at_eos=True)
    out_path = HEATMAP_DIR / f'attention_sample_{i + 1}.png'
    save_heatmap(attention[i], src_tokens, generated_tokens, out_path)
    print(f'Saved {out_path}')
    display(Image.open(out_path))


In [ ]:
# Summary: T1.2 vs T1.3
import pandas as pd

summary_rows = []
if 'BASELINE_METRICS' in globals():
    summary_rows.append({'Model': 'T1.2 Baseline GRU', **BASELINE_METRICS})
if 'ATTENTION_METRICS' in globals():
    summary_rows.append({'Model': 'T1.3 Bahdanau Attention', **ATTENTION_METRICS})

if summary_rows:
    metrics_df = pd.DataFrame(summary_rows)
    display(metrics_df)
else:
    print('No metrics found yet. Run the inference cells first.')

train_time_rows = []
if 'BASELINE_TRAIN_SECONDS' in globals():
    train_time_rows.append({'Model': 'T1.2 Baseline GRU', 'Train minutes this run': BASELINE_TRAIN_SECONDS / 60})
if 'ATTENTION_TRAIN_SECONDS' in globals():
    train_time_rows.append({'Model': 'T1.3 Bahdanau Attention', 'Train minutes this run': ATTENTION_TRAIN_SECONDS / 60})

if train_time_rows:
    time_df = pd.DataFrame(train_time_rows)
    display(time_df)
else:
    print('No training-time variables found yet. Run the training cells first.')

# Rough unit estimate: Colab units vary by account and runtime availability.
# As a planning number, L4 often behaves around 4-6 units/hour.
total_seconds = globals().get('BASELINE_TRAIN_SECONDS', 0) + globals().get('ATTENTION_TRAIN_SECONDS', 0)
estimated_units_low = total_seconds / 3600 * 4
estimated_units_high = total_seconds / 3600 * 6
print(f'Estimated L4 units for training time in this run: {estimated_units_low:.1f}-{estimated_units_high:.1f}')
print('Drive checkpoints: /content/drive/MyDrive/fit5217_a2/checkpoints/')
print('Predictions: /content/drive/MyDrive/fit5217_a2/outputs/predictions/')
print('Attention heatmaps: /content/drive/MyDrive/fit5217_a2/outputs/attention_heatmaps/')
